# 02 — Dimension Tables

**Project:** World Forest Change & GDP Correlation Analysis
**Database:** db_forestgdp (MSSQL Server)
**Team:** [Kucerova Kristina], [Kmetova Barbara]
**Date:** May 2026

## Purpose
This notebook builds the dimension tables in the `dbo` schema based on
findings from 01_raw_exploration. All data quality issues identified
in the exploration step are handled here.

## Schema structure
- `raw` — source tables loaded as-is from Kaggle
- `dbo` — cleaned dimension and fact tables

In [22]:
-- Create dim_country
CREATE TABLE dbo.dim_country (
    country_code    VARCHAR(50)     NOT NULL PRIMARY KEY,
    country_name    NVARCHAR(100),
    region          NVARCHAR(100),
    income_group    NVARCHAR(50)
);

Msg 2714, Level 16, State 6, Line 2
There is already an object named 'dim_country' in the database.

Total execution time: 00:00:00.020

In [3]:
-- Insert all countries that have a valid ISO3 code
INSERT INTO dbo.dim_country (country_code, country_name, region, income_group)
SELECT DISTINCT
    f.Code          AS country_code,
    f.Country       AS country_name,
    i.region,
    i.income_group
FROM raw.Forest_year f
LEFT JOIN raw.stg_gdp_income i ON i.country_code = f.Code
WHERE f.Code IS NOT NULL AND f.Code <> '';

(214 rows affected)

Total execution time: 00:00:00.110

In [5]:
-- Insert entities that have no ISO3 code
-- CONCAT builds AGG_ prefix e.g. AGG_European_Union
INSERT INTO dbo.dim_country (country_code, country_name, region, income_group)
SELECT DISTINCT
    CONCAT('AGG_', REPLACE(f.Country, ' ', '_')) AS country_code,
    f.Country       AS country_name,
    ig.region,
    ig.income_group
FROM raw.Forest_year f
LEFT JOIN raw.stg_gdp_income ig ON ig.country_code = f.Code
WHERE f.Code IS NULL OR f.Code = '';

(7 rows affected)

Total execution time: 00:00:00.040

In [6]:
-- Create dim_year
CREATE TABLE dbo.dim_year (year INT NOT NULL PRIMARY KEY);

Commands completed successfully.

Total execution time: 00:00:00.032

In [7]:
-- Creat dim_year with all years present in Forest_year (the largest date range)
INSERT INTO dbo.dim_year (year)
SELECT DISTINCT Year
FROM raw.Forest_year
ORDER BY Year;

(120 rows affected)

Total execution time: 00:00:00.039

In [25]:
-- Add subregion attribute to dim_country
ALTER TABLE dbo.dim_country
ADD subregion NVARCHAR(100);

Commands completed successfully.

Total execution time: 00:00:00.026

In [ ]:
-- Populate subregion from raw.Forest_Policy_Legislation joined on iso3
--Adding subregion to dim_country
--Subregion attribute added from raw.forest_policy joined on iso3.
--Provides more granular geographic grouping for regional analysis alongside the existing region column from stg_gdp_income.
UPDATE c
SET c.subregion = p.subregions
FROM dbo.dim_country c
JOIN raw.Forest_Policy_Legislation p ON p.iso3 = c.country_code;

(211 rows affected)

Total execution time: 00:00:00.987

In [27]:
-- Check subregion populated correctly
SELECT country_code, country_name, region, subregion
FROM dbo.dim_country
WHERE subregion IS NOT NULL
ORDER BY region, subregion;

(211 rows affected)

country_code | country_name                     | region                     | subregion                  
-------------+----------------------------------+----------------------------+----------------------------
GUF          | French Guiana                    | NULL                       |                            
ESH          | Western Sahara                   | NULL                       | Northern Africa            
FJI          | Fiji                             | East Asia & Pacific        |                            
FSM          | Micronesia (country)             | East Asia & Pacific        |                            
ASM          | American Samoa                   | East Asia & Pacific        |                            
AUS          | Australia                        | East Asia & Pacific        |                            
GUM          | Guam                             | East Asia & Pacific        |                            
KIR          | K

In [29]:
-- Manually set region and subregion for French Guiana
-- French overseas territory in South America, not in WB or FAO policy data
UPDATE dbo.dim_country
SET region    = 'Latin America & Caribbean',
    subregion = 'South America'
WHERE country_code = 'GUF';

(1 row affected)

Total execution time: 00:00:00.019

In [37]:
UPDATE dbo.dim_country
SET subregion = NULL
WHERE subregion = '';

(75 rows affected)

Total execution time: 00:00:00.992

In [38]:
UPDATE c
SET c.subregion = p.subregions
FROM dbo.dim_country c
JOIN raw.Forest_Policy_Legislation p ON p.iso3 = c.country_code
WHERE p.subregions IS NOT NULL 
AND p.subregions <> '';

(135 rows affected)

Total execution time: 00:00:00.574

In [41]:
SELECT iso3, name, regions, subregions
FROM raw.Forest_change_reason
WHERE iso3= 'ESP';

(24 rows affected)

iso3 | name  | regions | subregions
-----+-------+---------+-----------
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
ESP  | Spain | Europe  |           
(24 rows)

Total execution time: 00:00:00.91